In [69]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [71]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [72]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'B-1Density',
        'B-2Density',
        'B0Share',
        'B1Share',
        'B-1Share',
        'B-2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare'
    ]


In [73]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [74]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Weekly'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [75]:
print(query)


CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WHEN kd.KPIId = 'FOVMain' THEN kd.Value END) AS [FOVMain],
    SUM(CASE WHEN kd.KPIId = 'SurveyDurationHours' THEN kd.Va

In [77]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026 AND BoundaryRegion IS NULL;"
df = pd.read_sql_query(query, conn)
conn.close()